In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional

torch.manual_seed(42)
np.random.seed(42)

print("✅ Imports successful!")

### Before Attention: The Bottleneck Problem

**Traditional Sequence Models (RNNs):**
```
Input:  [The, cat, sat, on, the, mat]
         ↓    ↓    ↓   ↓    ↓    ↓
RNN:    h1 → h2 → h3 → h4 → h5 → h6  ← Final hidden state
                                     ↓
                               Bottleneck!
```

**Problems:**
1. All information compressed into a fixed-size vector
2. Long-range dependencies are lost
3. Sequential processing (can't parallelize)

**Attention Solution:**
```
Input:  [The, cat, sat, on, the, mat]
         ↓    ↓    ↓   ↓    ↓    ↓
        h1   h2   h3  h4   h5   h6
         ↘   ↓   ↙ ↘  ↓   ↙
           Attention
             ↓
        Weighted sum of ALL inputs!
```

**Benefits:**
1. Access to all inputs directly
2. Learns what to focus on
3. Can process in parallel

---

## 2. Scaled Dot-Product Attention

The core attention mechanism:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- **Q (Query)**: What am I looking for?
- **K (Key)**: What do I have to offer?
- **V (Value)**: The actual information
- **$d_k$**: Dimension of keys (for scaling)

### Intuition

Think of it like a database lookup:
1. **Query**: "Show me documents about 'cats'"
2. **Keys**: Document titles/descriptions
3. **Values**: Actual document content
4. **Attention weights**: Similarity between query and each key
5. **Output**: Weighted average of values based on similarity

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    Args:
        Q: Query matrix (batch_size, seq_len, d_k)
        K: Key matrix (batch_size, seq_len, d_k)
        V: Value matrix (batch_size, seq_len, d_v)
        mask: Optional mask (batch_size, seq_len, seq_len)
    
    Returns:
        output: Attention output (batch_size, seq_len, d_v)
        attention_weights: Attention weights (batch_size, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    
    # 1. Compute attention scores: Q·K^T
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, seq, seq)
    
    # 2. Scale by sqrt(d_k) to prevent softmax saturation
    scores = scores / np.sqrt(d_k)
    
    # 3. Apply mask (optional) - set masked positions to -inf
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # 4. Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)  # (batch, seq, seq)
    
    # 5. Weighted sum of values
    output = torch.matmul(attention_weights, V)  # (batch, seq, d_v)
    
    return output, attention_weights

# Example: Simple attention
batch_size = 1
seq_len = 4
d_k = 8

Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, attention_weights = scaled_dot_product_attention(Q, K, V)

print("Input shapes:")
print(f"  Q: {Q.shape}")
print(f"  K: {K.shape}")
print(f"  V: {V.shape}")
print(f"\nOutput shapes:")
print(f"  Output: {output.shape}")
print(f"  Attention weights: {attention_weights.shape}")
print(f"\nAttention weights (row i = how much position i attends to each position):")
print(attention_weights[0].detach().numpy())
print(f"\n✅ Each row sums to 1: {attention_weights[0].sum(dim=-1)}")

### Visualizing Attention

In [ ]:
def visualize_attention(attention_weights, words=None):
    """
    Visualize attention weights as a heatmap
    """
    # Get attention matrix (seq_len, seq_len)
    att = attention_weights[0].detach().numpy()
    
    # Default labels
    if words is None:
        words = [f"Token {i+1}" for i in range(att.shape[0])]
    
    # Plot heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(att, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=words, yticklabels=words,
                cbar_kws={'label': 'Attention Weight'})
    plt.xlabel('Keys (attending to)')
    plt.ylabel('Queries (attending from)')
    plt.title('Attention Weights Heatmap')
    plt.tight_layout()
    plt.show()

# Visualize
visualize_attention(attention_weights, words=['The', 'cat', 'sat', 'down'])

---

## 3. Self-Attention Example - Understanding Context

In [ ]:
class SelfAttention(nn.Module):
    """
    Self-Attention Layer
    Learns to create Q, K, V from the same input
    """
    
    def __init__(self, embed_dim):
        super().__init__()
        
        # Linear projections for Q, K, V
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        
        self.embed_dim = embed_dim
    
    def forward(self, x, mask=None):
        """
        Args:
            x: Input (batch_size, seq_len, embed_dim)
            mask: Optional mask
        
        Returns:
            output: Attention output
            attention_weights: Attention weights
        """
        # Project to Q, K, V
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        
        # Apply attention
        output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        return output, attention_weights

# Example: Sentence with ambiguous word
# "The bank can guarantee deposits will eventually cover future tuition costs"
# Word "bank" - is it financial institution or river bank?

# Simulate word embeddings
embed_dim = 16
seq_len = 5
words = ['The', 'bank', 'guarantees', 'deposits', 'money']

# Create embeddings (in real NLP, these come from word2vec, BERT, etc.)
embeddings = torch.randn(1, seq_len, embed_dim)

# Apply self-attention
self_attn = SelfAttention(embed_dim)
output, attention_weights = self_attn(embeddings)

print(f"Input shape: {embeddings.shape}")
print(f"Output shape: {output.shape}")
print(f"\nAttention pattern for 'bank' (position 1):")
print(f"  Attends to 'The': {attention_weights[0, 1, 0]:.3f}")
print(f"  Attends to 'bank': {attention_weights[0, 1, 1]:.3f}")
print(f"  Attends to 'guarantees': {attention_weights[0, 1, 2]:.3f}")
print(f"  Attends to 'deposits': {attention_weights[0, 1, 3]:.3f}")
print(f"  Attends to 'money': {attention_weights[0, 1, 4]:.3f}")

# Visualize
visualize_attention(attention_weights, words=words)

---

## 4. Multi-Head Attention

**Idea**: Run multiple attention mechanisms in parallel, each learning different patterns!

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

where:
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**Benefits:**
1. Different heads can focus on different aspects
2. One head might learn syntax, another semantics
3. More expressive power

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention Layer
    """
    
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Linear projections for all heads (combined)
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        
        # Output projection
        self.out = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: Input (batch_size, seq_len, embed_dim)
        
        Returns:
            output: Multi-head attention output
            attention_weights: List of attention weights per head
        """
        batch_size, seq_len, embed_dim = x.size()
        
        # 1. Linear projections
        Q = self.query(x)  # (batch, seq, embed_dim)
        K = self.key(x)
        V = self.value(x)
        
        # 2. Split into multiple heads
        # Reshape: (batch, seq, embed_dim) → (batch, seq, num_heads, head_dim)
        # Transpose: (batch, num_heads, seq, head_dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 3. Apply attention for each head
        d_k = self.head_dim
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_output = torch.matmul(attention_weights, V)
        
        # 4. Concatenate heads
        # Transpose: (batch, num_heads, seq, head_dim) → (batch, seq, num_heads, head_dim)
        attention_output = attention_output.transpose(1, 2).contiguous()
        # Reshape: (batch, seq, embed_dim)
        attention_output = attention_output.view(batch_size, seq_len, embed_dim)
        
        # 5. Final linear projection
        output = self.out(attention_output)
        
        return output, attention_weights

# Example
embed_dim = 64
num_heads = 8
seq_len = 10
batch_size = 2

mha = MultiHeadAttention(embed_dim, num_heads)
x = torch.randn(batch_size, seq_len, embed_dim)
output, attention_weights = mha(x)

print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"  (batch_size, num_heads, seq_len, seq_len)")
print(f"\nNumber of parameters: {sum(p.numel() for p in mha.parameters()):,}")

### Visualizing Different Heads

In [ ]:
def visualize_multi_head_attention(attention_weights, num_heads=4):
    """
    Visualize attention patterns from different heads
    """
    fig, axes = plt.subplots(2, num_heads // 2, figsize=(15, 6))
    axes = axes.ravel()
    
    for i in range(num_heads):
        att = attention_weights[0, i].detach().numpy()  # (seq_len, seq_len)
        
        sns.heatmap(att, annot=False, cmap='viridis', ax=axes[i],
                   cbar=True, square=True)
        axes[i].set_title(f'Head {i+1}')
        axes[i].set_xlabel('Key')
        axes[i].set_ylabel('Query')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Observations:")
    print("  • Different heads show different patterns")
    print("  • Some heads might focus on local context")
    print("  • Others might capture long-range dependencies")
    print("  • This diversity makes the model more expressive!")

# Visualize
visualize_multi_head_attention(attention_weights, num_heads=8)

---

## 5. Masked Attention - For Autoregressive Models

**Use case**: Language models that generate text one word at a time

**Problem**: When predicting word $i$, we can't see words $i+1, i+2, ...$ (future words)

**Solution**: Mask future positions

In [ ]:
def create_causal_mask(seq_len):
    """
    Create causal (lower triangular) mask
    Prevents attending to future positions
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

# Example
seq_len = 5
mask = create_causal_mask(seq_len)

print("Causal mask:")
print(mask.numpy())
print("\n1 = can attend, 0 = cannot attend (future positions)")

# Visualize
plt.figure(figsize=(6, 5))
sns.heatmap(mask.numpy(), annot=True, fmt='g', cmap='Blues',
           cbar=False, square=True)
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Causal Mask\n(Autoregressive Generation)')
plt.tight_layout()
plt.show()

print("\n💡 Usage:")
print("  Position 0 can only see position 0")
print("  Position 1 can see positions 0-1")
print("  Position 2 can see positions 0-2")
print("  etc.")

In [ ]:
# Apply masked attention
embed_dim = 16
seq_len = 5
words = ['Once', 'upon', 'a', 'time', '<predict>']

x = torch.randn(1, seq_len, embed_dim)
mask = create_causal_mask(seq_len).unsqueeze(0).unsqueeze(0)  # (1, 1, seq, seq)

# Self-attention with mask
self_attn = SelfAttention(embed_dim)
output, attention_weights = self_attn(x, mask=mask)

print("Masked attention weights:")
visualize_attention(attention_weights, words=words)

print("\n✅ Notice: Upper triangle is all zeros (can't attend to future)!")

---

## 6. Cross-Attention - Connecting Two Sequences

**Use case**: Machine translation, image captioning

**Difference from self-attention**:
- **Self-attention**: Q, K, V all from same sequence
- **Cross-attention**: Q from one sequence, K and V from another

**Example**: Translating English to French
- Q: French words being generated
- K, V: English words (source sentence)

In [ ]:
class CrossAttention(nn.Module):
    """
    Cross-Attention Layer
    Query from one sequence, Key and Value from another
    """
    
    def __init__(self, embed_dim):
        super().__init__()
        
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, target, source, mask=None):
        """
        Args:
            target: Target sequence (batch, target_len, embed_dim)
            source: Source sequence (batch, source_len, embed_dim)
        
        Returns:
            output: Cross-attention output
            attention_weights: Attention from target to source
        """
        # Q from target, K and V from source
        Q = self.query(target)
        K = self.key(source)
        V = self.value(source)
        
        output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        return output, attention_weights

# Example: Translation
embed_dim = 16
source_len = 4  # English: "I love cats"
target_len = 3  # French: "J'aime les" (generating "chats")

english_words = ['I', 'love', 'cats', '<EOS>']
french_words = ["J'aime", 'les', '<predict>']

source = torch.randn(1, source_len, embed_dim)  # English embeddings
target = torch.randn(1, target_len, embed_dim)  # French embeddings

cross_attn = CrossAttention(embed_dim)
output, attention_weights = cross_attn(target, source)

print(f"Source (English): {source.shape}")
print(f"Target (French): {target.shape}")
print(f"Output: {output.shape}")
print(f"Attention weights: {attention_weights.shape}")
print("  (Each French word attends to all English words)")

# Visualize
plt.figure(figsize=(8, 6))
att = attention_weights[0].detach().numpy()
sns.heatmap(att, annot=True, fmt='.2f', cmap='Greens',
           xticklabels=english_words, yticklabels=french_words,
           cbar_kws={'label': 'Attention Weight'})
plt.xlabel('Source (English)')
plt.ylabel('Target (French)')
plt.title('Cross-Attention: French → English')
plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("  Row 0: When generating \"J'aime\", which English words to focus on?")
print("  Row 1: When generating \"les\", which English words to focus on?")
print("  Row 2: When generating next word, which English words to focus on?")

---

## 7. Practical Application - Sequence Classification

In [ ]:
class AttentionClassifier(nn.Module):
    """
    Simple classifier using self-attention
    """
    
    def __init__(self, vocab_size, embed_dim, num_heads, num_classes):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.fc = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        """
        Args:
            x: Token indices (batch, seq_len)
        
        Returns:
            logits: Class logits (batch, num_classes)
        """
        # Embed tokens
        x = self.embedding(x)  # (batch, seq, embed)
        
        # Apply attention
        x, _ = self.attention(x)  # (batch, seq, embed)
        
        # Pool: take mean across sequence
        x = x.mean(dim=1)  # (batch, embed)
        
        # Classify
        logits = self.fc(x)  # (batch, num_classes)
        
        return logits

# Example
vocab_size = 1000
embed_dim = 64
num_heads = 4
num_classes = 2  # Binary classification

model = AttentionClassifier(vocab_size, embed_dim, num_heads, num_classes)

# Dummy batch
batch_size = 4
seq_len = 20
x = torch.randint(0, vocab_size, (batch_size, seq_len))

logits = model(x)
print(f"Input shape: {x.shape}")
print(f"Output logits: {logits.shape}")
print(f"\nPredictions: {torch.argmax(logits, dim=1)}")

print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---

## Summary

### ✅ What You Learned

1. **Attention Mechanism**: Revolutionary approach to sequence modeling
2. **Scaled Dot-Product**: Core attention computation
3. **Self-Attention**: Learning relationships within a sequence
4. **Multi-Head Attention**: Multiple attention patterns in parallel
5. **Masked Attention**: For autoregressive generation
6. **Cross-Attention**: Connecting two different sequences
7. **Applications**: Classification, translation, generation

### 🔑 Key Formulas

**Attention:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Multi-Head:**
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

### 💡 Key Insights

1. **Q, K, V**: Think of them as query, database keys, and database values
2. **Softmax**: Ensures attention weights sum to 1 (weighted average)
3. **Scaling**: $\sqrt{d_k}$ prevents softmax saturation
4. **Multi-head**: Different heads learn different relationships
5. **Self-attention**: All positions can attend to all positions
6. **Cross-attention**: Connect encoder and decoder

### 🎯 What's Next?

**Next notebook:** `05_transformer_architecture.ipynb`

You'll learn:
- Complete Transformer architecture
- Positional encoding
- Encoder and Decoder stacks
- Training transformers
- Fine-tuning pre-trained models (BERT, GPT)

### 📚 Additional Resources

- [Attention Is All You Need (Paper)](https://arxiv.org/abs/1706.03762)
- [Illustrated Transformer](http://jalammar.github.io/illustrated-transformer/)
- [Attention Mechanism (Distill.pub)](https://distill.pub/2016/augmented-rnns/)

---

**Incredible work!** You now understand the attention mechanism that powers modern NLP models like BERT, GPT, and T5! 🎯